# Employetics — 40s Bold Kinetic Typography Trailer Ad (NVIDIA GPU)

Render the **40-second, 16:9, 1080p / 4K @ 60fps** trailer-style kinetic-typography commercial:
- **Palette:** Deep Navy (`#0F2A52`), Pure White (`#FFFFFF`), Pure Black (`#000000`), and One Amber Accent (`#F59E0B`) — **no neon colours, no people, no photos**
- **0–3 s (HOOK):** Black screen → *“YOUR BEST EMPLOYEE”* → *“JUST QUIT.”* with heavy impact & screen-shake
- **3–11 s (PROBLEM):** Deep Navy → *“LATE NIGHTS.”* *“LOST WEEKENDS.”* *“NOBODY NOTICED.”* + flat corner battery draining Green → Empty Red
- **11–14 s (REVEAL):** Fast bright wipe to white → bouncing *“MEET EMPLOYETICS.”* + typed *“AI THAT SPOTS BURNOUT BEFORE PEOPLE QUIT.”*
- **14–26 s (APP):** Flat laptop slides & 3D-spins forward with **100% plain solid navy (`#0F2A52`) screen** for dropping your screen recording + *“WHO’S AT RISK?”* & *“AND WHY.”* animated arrow callouts
- **26–34 s (SOLUTION):** Flat battery recharges Red → Full Green + slams *“ACT EARLY.”* → *“KEEP YOUR PEOPLE.”*
- **34–40 s (EDITOR OUTRO):** Everything collapses into a **100% pure solid white (`#FFFFFF`) screen** for adding your logo & slogan in your editor
- **Voice-Over (Deep Confident Male):** *“Your best employee just quit. Nobody saw it coming. Now you can.”* + deep bass hits & rising 128 BPM synth pulse

> **Before running:** Go to **Runtime → Change runtime type** and select **L4 GPU** or **T4 GPU**, then click **Runtime → Run all**.

In [ ]:
#@title 1. Configure Deep Male Voiceover & GPU Render Settings
GITHUB_REPO_URL = "https://github.com/abdullahmehmood1/EmployeticsAD.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

#@markdown ### Deep Confident Male Voiceover & Trailer Sound Design
VOICE_ENGINE = "edge" #@param ["edge", "upload_my_voiceover", "chatterbox", "elevenlabs", "kokoro"]
EDGE_VOICE = "en-US-BrianMultilingualNeural" #@param ["en-US-BrianMultilingualNeural", "en-US-ChristopherNeural", "en-US-AndrewMultilingualNeural", "en-GB-RyanNeural"]
KOKORO_VOICE = "am_fenrir" #@param ["am_fenrir", "am_michael", "am_adam", "bm_george"]
ELEVENLABS_VOICE = "brian" #@param ["brian", "adam", "chris"]
ELEVENLABS_API_KEY = "" #@param {type:"string"}
VOICE_SPEED = 0.94 #@param {type:"slider", min:0.85, max:1.10, step:0.01}
MUSIC_VOLUME = 0.58 #@param {type:"slider", min:0.0, max:0.85, step:0.02}

#@markdown ### Video Output Settings
BURN_CAPTIONS = False #@param {type:"boolean"}
RESOLUTION = "1080p (1920x1080 @ 60fps)" #@param ["1080p (1920x1080 @ 60fps)", "4K (3840x2160 @ 60fps)"]
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}

SCALE = 2.0 if RESOLUTION.startswith("4K") else 1.0
OUTPUT_FILENAME = "employetics-kinetic-ad-4k60.mp4" if SCALE >= 2.0 else "employetics-kinetic-ad-1080p60.mp4"
print(f"Configured: {RESOLUTION} | Voice Engine={VOICE_ENGINE} | Voice={EDGE_VOICE} | Speed={VOICE_SPEED} | Trailer SFX/Pulse={MUSIC_VOLUME}")


In [ ]:
#@title 2. Verify NVIDIA CUDA GPU & Install Clean Dependencies (Ubuntu 24.04 Noble Ready)
import os, shutil, subprocess, torch

print("=== 1) Verifying NVIDIA CUDA GPU & NVENC ===")
if torch.cuda.is_available():
    print(f"CUDA GPU ACTIVE: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")
else:
    print("WARNING: No CUDA GPU detected! Go to Runtime -> Change runtime type -> L4 or T4 GPU!")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

print("\n=== 2) Installing minimal system packages (Ubuntu 24.04 Noble compatible) ===")
!apt-get -qq update >/dev/null 2>&1 || true
!apt-get -qq install -y ffmpeg espeak-ng libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2t64 libpango-1.0-0 libcairo2 || \
  apt-get -qq install -y ffmpeg espeak-ng libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 libcairo2

print("\n=== 3) Installing Node.js 22 ===")
node_ver = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_ver.returncode != 0 or not node_ver.stdout.strip().startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1
    !apt-get -qq install -y nodejs
!node -v && npm -v

print("\n=== 4) Cloning / Updating Repository ===")
PROJECT_DIR = "/content/employeticsAD"
if os.path.exists(os.path.join(PROJECT_DIR, ".git")):
    !git -C {PROJECT_DIR} fetch origin && git -C {PROJECT_DIR} checkout {BRANCH} && git -C {PROJECT_DIR} reset --hard origin/{BRANCH}
else:
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {PROJECT_DIR}

%cd /content/employeticsAD

print("\n=== 5) Installing Python Voiceover & Audio Packages (preserving Colab pins) ===")
!pip install -q edge-tts soundfile
if VOICE_ENGINE == "kokoro":
    !pip install -q "kokoro>=0.9.4" "misaki[en]>=0.9.4"
    !python -m spacy download en_core_web_sm -q || true
elif VOICE_ENGINE == "chatterbox":
    !pip install -q --no-deps chatterbox-tts
    !pip install -q librosa s3tokenizer resemble-perth conformer safetensors omegaconf diffusers

print("\n=== 6) Installing Remotion & Bundled Headless Browser ===")
!npm install --no-audit --no-fund
!npx remotion browser ensure
print("\nSetup complete with ZERO errors!")


In [ ]:
#@title 3. Generate (or Upload) Deep Male Voiceover + Heavy Bass Hits & Rising Synth Pulse
%cd /content/employeticsAD
import os, subprocess
from IPython.display import Audio, display

if VOICE_ENGINE == "upload_my_voiceover":
    from google.colab import files
    print("Upload your voiceover MP3 or WAV file below:")
    uploaded = files.upload()
    assert len(uploaded) > 0, "No audio file uploaded!"
    vo_filename = list(uploaded.keys())[0]
    vo_path = os.path.abspath(vo_filename)
    cmd = [
        "python", "scripts/generate_audio.py",
        "--engine", "custom",
        "--custom-voiceover", vo_path,
        "--music-volume", str(MUSIC_VOLUME),
    ]
else:
    cmd = [
        "python", "scripts/generate_audio.py",
        "--engine", VOICE_ENGINE,
        "--edge-voice", EDGE_VOICE,
        "--kokoro-voice", KOKORO_VOICE,
        "--elevenlabs-voice", ELEVENLABS_VOICE,
        "--speed", str(VOICE_SPEED),
        "--music-volume", str(MUSIC_VOLUME),
    ]
    if ELEVENLABS_API_KEY.strip():
        cmd.extend(["--elevenlabs-key", ELEVENLABS_API_KEY.strip()])

res = subprocess.run(cmd)
assert res.returncode == 0, "Audio generation failed! See logs above."
assert os.path.exists("public/audio/master_audio.wav"), "master_audio.wav was not generated!"

print("\n1) Isolated Deep Male Voiceover Preview:")
display(Audio("public/audio/voiceover.wav", autoplay=False))
print("\n2) Final Broadcast Master Mix (-14 LUFS: Voiceover + Deep Bass Hits + Rising Synth Pulse):")
display(Audio("public/audio/master_audio.wav", autoplay=False))


In [ ]:
#@title 4. Render 40-Second Kinetic Trailer on NVIDIA GPU (CUDA Motion-Blur + Direct NVENC)
%cd /content/employeticsAD

caption_flag = "--captions" if BURN_CAPTIONS else ""
out_path = f"out/{OUTPUT_FILENAME}"

!python scripts/render_video.py \
  --output {out_path} \
  --scale {SCALE} \
  --crf 16 \
  {caption_flag}


In [ ]:
#@title 5. Preview Rendered Video Inline (Zero-Blink + Full Trailer Audio)
from IPython.display import Video, display
import os

video_path = f"/content/employeticsAD/out/{OUTPUT_FILENAME}"
assert os.path.exists(video_path), f"Video not found at {video_path}"
print(f"Rendered video size: {os.path.getsize(video_path) / (1024*1024):.2f} MB")
display(Video(video_path, embed=True, width=960))


In [ ]:
#@title 6. Download MP4 + SRT Captions (and Optional Google Drive Backup)
import os, shutil
from google.colab import files

video_path = f"/content/employeticsAD/out/{OUTPUT_FILENAME}"
srt_path = "/content/employeticsAD/public/audio/captions.srt"

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest_dir = "/content/drive/MyDrive/EmployeticsAd"
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy2(video_path, os.path.join(dest_dir, OUTPUT_FILENAME))
    if os.path.exists(srt_path):
        shutil.copy2(srt_path, os.path.join(dest_dir, "captions.srt"))
    print(f"Saved copies to Google Drive: {dest_dir}")

files.download(video_path)
if os.path.exists(srt_path):
    files.download(srt_path)
